# 2.12 · 蒙特卡洛方法 / Monte Carlo Methods —— Part 2 收官 🏁

> **课程定位**
> Part 2 最后一课，也是**全 Part 的"统一场论"**：CLT 模拟（2.3）、覆盖率审计（2.5）、p 值验证（2.6）、模拟法 power（2.8）、贝叶斯 A/B 采样（2.10）、bootstrap（2.11）——**全是蒙特卡洛的特例**。本课补齐它的理论身份证 + 采样工具箱 + MCMC 预告。
> The unifying theory of Part 2: everything we've simulated was Monte Carlo. This lesson issues its theoretical ID card + the sampling toolbox + an MCMC teaser.

> 💡 **面试相关**
> - "用随机数估 π / 求积分" ★★★★（手写代码题）
> - "逆变换采样原理" ★★★（量化岗）
> - "重要性采样什么时候用" ★★★
> - "MCMC 直觉一句话" ★★★（为 Part 18 打底）

---

## 目录
1. [核心恒等式：期望 = 平均 ⭐](#1)
2. [开胃菜：估 π 与收敛速度](#2)
3. [MC 积分：维度诅咒的克星 ⭐](#3)
4. [怎么造样本 (1)：逆变换采样 ⭐](#4)
5. [怎么造样本 (2)：拒绝采样](#5)
6. [重要性采样：稀有事件的救星 ⭐](#6)
7. [方差削减：对偶变量](#7)
8. [MCMC 预告：Metropolis 50 行](#8)
9. [实战：用 MC 给真实业务问题定价](#9)
10. [小结 + Part 2 总结 🏁](#10)


<a id="1"></a>
## 1. 核心恒等式：期望 = 平均 ⭐ / The Core Identity

想算的一切几乎都能写成期望：
$$I = \mathbb{E}_{p}[f(X)] = \int f(x)\,p(x)\,dx$$

**蒙特卡洛估计**：抽 $N$ 个样本，取平均：
$$\hat{I}_N = \frac{1}{N}\sum_{i=1}^N f(x_i), \qquad x_i \sim p$$

理论保障全部来自 Part 2 前半：
- **LLN**（2.3）：$\hat{I}_N \to I$ —— 一定收敛
- **CLT**（2.3）：$\mathrm{SE}(\hat{I}_N) = \sigma_f / \sqrt{N}$ —— 收敛速度 + **免费的误差条**

**概率也是期望**（指示函数）：$\Pr(A) = \mathbb{E}[\mathbb{1}\{A\}]$ —— 所以"数命中比例"就是 MC。
Probabilities are expectations of indicators — counting hits IS Monte Carlo.


<a id="2"></a>
## 2. 开胃菜：估 π 与收敛速度 / Estimating π

单位正方形内随机撒点，落入内切圆的比例 ≈ $\pi/4$：
$$\pi \approx 4 \cdot \frac{\#\{x_i^2 + y_i^2 \le 1\}}{N}$$


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 撒点 + 收敛曲线 / Darts + the convergence curve
N = 200_000
pts = rng.uniform(-1, 1, (N, 2))
inside = (pts**2).sum(axis=1) <= 1
pi_running = 4 * np.cumsum(inside) / np.arange(1, N+1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
show = 3000
axes[0].scatter(pts[:show, 0], pts[:show, 1], c=inside[:show], cmap="coolwarm", s=2)
axes[0].set_aspect("equal"); axes[0].set_title(f"3000 darts → π̂ = {4*inside[:show].mean():.4f}")

ns = np.arange(1, N+1)
axes[1].plot(ns, np.abs(pi_running - np.pi), lw=0.6)
axes[1].plot(ns, 4*np.sqrt(np.pi/4*(1-np.pi/4))/np.sqrt(ns), "r--", lw=1.5, label="理论 SE ∝ 1/√N")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("N"); axes[1].set_ylabel("|error|"); axes[1].legend()
axes[1].set_title("Error shrinks at exactly 1/√N")
plt.tight_layout(); plt.show()
print(f"N={N:,}: π̂ = {pi_running[-1]:.5f}  (真值 {np.pi:.5f})")


**log-log 图上误差贴着 $1/\sqrt{N}$ 直线下滑**——2.3 的 √n 法则第三次现身。**多一位小数精度 = 100 倍样本**：MC 是"快糙猛"不是"高精尖"。
The same sqrt-N law, third appearance. One more digit of precision costs 100x samples — MC is robust, not precise.


<a id="3"></a>
## 3. MC 积分：维度诅咒的克星 ⭐ / MC Integration vs the Curse

确定性数值积分（梯形/Simpson）误差 $O(N^{-k/d})$——**$d$ 个维度均摊网格点**，高维瞬间饿死。

**MC 的奇迹**：$\mathrm{SE} = \sigma_f/\sqrt{N}$ **与维度 $d$ 无关**！

| 维度 | 网格法 ($N=10^6$ 点) | MC ($N=10^6$) |
|---|---|---|
| d=1 | 每维 $10^6$ 点，误差极小 | SE ∝ 1/1000 |
| d=10 | 每维只剩 4 点（$4^{10}≈10^6$）→ 垃圾 | **SE 还是 ∝ 1/1000** |
| d=100 | 每维 1 点出头 → 不可用 | **照常工作** |

这就是金融（百维投资组合）、统计物理、贝叶斯推断（百万参数后验）全靠 MC 的原因。
This dimension-independence is why finance, physics, and Bayesian inference all run on MC.


In [ ]:
# 10 维积分: E[exp(-‖x‖²)] over [0,1]^10 / A 10-D integral
d = 10
f = lambda x: np.exp(-np.sum(x**2, axis=-1))

# 真值 (可分离: 每维 ∫₀¹e^{-t²}dt = √π/2·erf(1)) / separable truth
from scipy.special import erf
truth = (np.sqrt(np.pi)/2 * erf(1))**d

for N_ in [1_000, 100_000, 10_000_000]:
    x = rng.uniform(0, 1, (N_, d))
    vals = f(x)
    est, se = vals.mean(), vals.std(ddof=1)/np.sqrt(N_)
    print(f"N={N_:>10,}: Î = {est:.6f} ± {se:.6f}   (真值 {truth:.6f}, "
          f"偏 {abs(est-truth)/se:.1f} SE)")


**注意附带的 ±SE**——MC 永远自带误差条（CLT 白送），这是网格法没有的福利。
Note the free error bars — the CLT throws them in at no charge.


<a id="4"></a>
## 4. 怎么造样本 (1)：逆变换采样 ⭐ / Inverse Transform

MC 需要"从 $p$ 抽样"。**最基本的武器**：

$$U \sim \mathrm{Uniform}(0,1) \;\Rightarrow\; X = F^{-1}(U) \sim F$$

**证明一行**：$\Pr(X \le x) = \Pr(F^{-1}(U) \le x) = \Pr(U \le F(x)) = F(x)$ ∎

2.2 节说 `ppf` 是被低估的主力——这就是它的高光时刻：**ppf = $F^{-1}$ = 采样器**。


In [ ]:
# 手写指数分布采样器 / Hand-rolled exponential sampler
# F(x) = 1-e^{-λx}  →  F⁻¹(u) = -ln(1-u)/λ
lam = 2.0
u = rng.uniform(0, 1, 100_000)
x_inv = -np.log(1 - u) / lam

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(x_inv, bins=100, density=True, alpha=0.7, label="inverse-transform samples")
xs = np.linspace(0, 4, 200)
ax.plot(xs, lam*np.exp(-lam*xs), "r-", lw=2, label=f"Exp({lam}) pdf")
ax.legend(); ax.set_title("X = F⁻¹(U): uniform noise in, exponential out")
plt.tight_layout(); plt.show()

print(f"样本均值 = {x_inv.mean():.4f}  (理论 1/λ = {1/lam})")
print(f"KS 检验 p = {st.kstest(x_inv, 'expon', args=(0, 1/lam)).pvalue:.3f}  (>0.05 ✓)")


<a id="5"></a>
## 5. 怎么造样本 (2)：拒绝采样 / Rejection Sampling

$F^{-1}$ 写不出来（如 Beta、截断分布）？**拒绝采样**：

1. 找一个好抽的"罩子"分布 $q$ 和常数 $M$ 使 $p(x) \le M q(x)$
2. 抽 $x \sim q$，$u \sim U(0,1)$
3. 若 $u \le \dfrac{p(x)}{M q(x)}$ 接受，否则扔掉重来

**几何直觉**：在 $Mq$ 的曲线下均匀撒点，只留 $p$ 曲线下方的——留下的点的 x 坐标就服从 $p$。**接受率 = $1/M$** → 罩子越贴身越高效。
Throw darts under the envelope; keep those under p. Acceptance rate = 1/M — tighter envelopes waste less.


In [ ]:
# 拒绝采样 Beta(2.7, 6.3), 罩子用 Uniform / Rejection-sample a Beta with a uniform envelope
a_, b_ = 2.7, 6.3
target = st.beta(a_, b_)
M = target.pdf(np.linspace(0.01, 0.99, 500)).max() * 1.02   # 罩子高度 / envelope height

N_try = 100_000
x_cand = rng.uniform(0, 1, N_try)
u = rng.uniform(0, 1, N_try)
accepted = x_cand[u <= target.pdf(x_cand) / M]

print(f"接受率: {len(accepted)/N_try:.1%}  (理论 1/M = {1/M:.1%})")
print(f"KS p = {st.kstest(accepted, 'beta', args=(a_, b_)).pvalue:.3f} ✓")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(accepted, bins=80, density=True, alpha=0.7, label=f"accepted ({len(accepted):,})")
xs = np.linspace(0, 1, 300)
ax.plot(xs, target.pdf(xs), "r-", lw=2, label="Beta(2.7, 6.3)")
ax.axhline(M, color="gray", ls=":", label=f"envelope M={M:.2f}")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


⚠ **高维的死刑**：接受率随维度**指数衰减**（罩子和目标的体积比越来越悬殊）——高维采样必须换 MCMC（第 8 节）。
Acceptance decays exponentially with dimension — high-D sampling needs MCMC.


<a id="6"></a>
## 6. 重要性采样：稀有事件的救星 ⭐ / Importance Sampling

**问题**：估 $\Pr(X > 5)$，$X \sim \mathcal{N}(0,1)$（真值 ≈ 2.9e-7）。朴素 MC 平均**抽 350 万个才命中 1 个**——SE 巨大。

**解法**：从"瞄准稀有区"的提议分布 $q$ 抽样，再用**权重**修正：
$$\mathbb{E}_p[f(X)] = \mathbb{E}_q\Big[f(X)\,\underbrace{\frac{p(X)}{q(X)}}_{w(X)}\Big]$$


In [ ]:
# 朴素 MC vs 重要性采样 / Naive vs importance sampling
truth = st.norm.sf(5)                       # 2.87e-7
N_ = 100_000

# 朴素: 几乎全 miss / Naive: almost no hits
x_naive = rng.standard_normal(N_)
est_naive = (x_naive > 5).mean()

# IS: 提议分布挪到稀有区 N(5, 1) / Proposal centered at the rare region
x_is = rng.normal(5, 1, N_)
w = st.norm.pdf(x_is) / st.norm.pdf(x_is, 5, 1)        # p/q
vals = (x_is > 5) * w
est_is, se_is = vals.mean(), vals.std(ddof=1)/np.sqrt(N_)

print(f"真值          = {truth:.3e}")
print(f"朴素 MC       = {est_naive:.3e}   (命中 {int((x_naive>5).sum())} 次 — 基本瞎猜)")
print(f"重要性采样    = {est_is:.3e} ± {se_is:.1e}   ← 同样 N, 误差小几个数量级")
print(f"\n等效样本效率: IS 用 1e5 样本 ≈ 朴素 MC 用 {truth*(1-truth)/se_is**2:.1e} 样本")


**同样 10 万样本，IS 把不可估变成三位有效数字**。应用：金融风险（尾部损失）、可靠性工程（罕见故障）、强化学习的 off-policy 评估（Part 17——"用旧策略的数据评估新策略"就是 IS 权重）。

⚠ **权重退化**：$q$ 选得差 → 个别权重巨大 → 方差爆炸。诊断：有效样本量 $\mathrm{ESS} = (\sum w)^2 / \sum w^2$。
Off-policy RL evaluation IS importance sampling. Watch for weight degeneracy — diagnose with ESS.


<a id="7"></a>
## 7. 方差削减：对偶变量 / Antithetic Variates

**免费午餐系列**：抽 $U$ 时同时用 $1-U$（完美负相关），成对平均：
$$\mathrm{Var}\Big(\frac{f(U) + f(1-U)}{2}\Big) = \frac{\mathrm{Var}(f)}{2}\,(1 + \rho), \quad \rho < 0 \Rightarrow \text{省样本}$$

$f$ 单调时 $\rho < 0$ 有保证。同族技术：control variates（用已知期望的相关量做"锚"）、分层抽样（2.4 的老朋友在 MC 里再就业）。


In [ ]:
# 对偶变量实测: 估 E[e^U] / Antithetic in action
truth_e = np.e - 1                          # ∫₀¹eᵘdu
N_pairs = 50_000

u1 = rng.uniform(0, 1, 2*N_pairs)           # 朴素: 2N 个独立样本
naive_se = np.exp(u1).std(ddof=1) / np.sqrt(2*N_pairs)

u2 = rng.uniform(0, 1, N_pairs)             # 对偶: N 对 (U, 1-U)
pairs = (np.exp(u2) + np.exp(1-u2)) / 2
anti_se = pairs.std(ddof=1) / np.sqrt(N_pairs)

print(f"同样 {2*N_pairs:,} 次函数求值:")
print(f"  朴素 SE = {naive_se:.5f}")
print(f"  对偶 SE = {anti_se:.5f}   ← 方差省 {(1-(anti_se/naive_se)**2)*100:.0f}%")


<a id="8"></a>
## 8. MCMC 预告：Metropolis 50 行 / The Metropolis Teaser

**终极问题**：后验 $p(\theta \mid \mathcal{D}) \propto \mathcal{L}(\theta)\,p(\theta)$ **只知道到比例常数**（分母积分算不动）且高维——逆变换/拒绝采样全跪。

**Metropolis (1953)**：构造一条**马尔可夫链**，让它的平稳分布恰好是目标：
1. 当前位置 $\theta$，提议 $\theta' = \theta + \mathcal{N}(0, s^2)$
2. 以概率 $\min\!\big(1, \frac{p(\theta')}{p(\theta)}\big)$ 接受——**只需要比值，归一化常数约掉了** ⭐
3. 重复。烧掉开头（burn-in），剩下的就是目标分布的（相关）样本


In [ ]:
def metropolis(log_target, theta0, n_steps, step, rng):
    chain = np.empty(n_steps); chain[0] = theta0
    lp = log_target(theta0)
    n_acc = 0
    for t in range(1, n_steps):
        prop = chain[t-1] + rng.normal(0, step)
        lp_prop = log_target(prop)
        if np.log(rng.uniform()) < lp_prop - lp:       # log 域避免下溢
            chain[t], lp = prop, lp_prop; n_acc += 1
        else:
            chain[t] = chain[t-1]
    return chain, n_acc/n_steps

# 目标: 一个双峰分布 (拒绝采样难配罩子, MCMC 无所谓)
log_bimodal = lambda x: np.log(0.4*st.norm.pdf(x, -2, 0.7) + 0.6*st.norm.pdf(x, 2.5, 1.0) + 1e-300)

chain, acc = metropolis(log_bimodal, theta0=0.0, n_steps=60_000, step=2.2, rng=rng)
chain = chain[5000:]                                    # burn-in

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(chain[:3000], lw=0.5); axes[0].set_title(f"trace (acc={acc:.0%}) — 在两峰间跳跃")
xs = np.linspace(-5, 6, 300)
axes[1].hist(chain, bins=100, density=True, alpha=0.7, label="MCMC samples")
axes[1].plot(xs, np.exp([log_bimodal(x) for x in xs]), "r-", lw=2, label="target")
axes[1].legend(); axes[1].set_title("Samples match the bimodal target")
plt.tight_layout(); plt.show()


**30 行代码采样任意"只知道形状"的分布**——这就是现代贝叶斯统计的发动机。正篇在 **Part 18**（调参、诊断、HMC/NUTS、PyMC）。

> 💡 面试一句话："MCMC builds a Markov chain whose stationary distribution is the target; the Metropolis ratio only needs the unnormalized density, which is why it works for posteriors."


<a id="9"></a>
## 9. 实战：用 MC 给真实业务问题定价 / Hands-on: Pricing a Business Decision

**场景**：订阅产品考虑"30 天无理由退款"政策。已知（带不确定性的）参数：
- 退款率 $r \sim \mathrm{Beta}(30, 270)$（历史 A/B 估计，~10%）
- 政策带来的转化提升 $l \sim \mathcal{N}(8\%, 2\%^2)$
- 客单价 \$40，月新客基数 10,000

**问题**：净收入变化的分布？亏钱的概率？——参数不确定性 × 非线性组合，**解析没戏，MC 三行**。
Uncertain parameters composed nonlinearly — analytics hopeless, MC trivial.


In [ ]:
S = 500_000
r = st.beta(30, 270).rvs(S, random_state=1)            # 退款率
lift = rng.normal(0.08, 0.02, S)                        # 转化提升
base_customers, price = 10_000, 40.0

new_customers = base_customers * (1 + lift)
revenue_new = new_customers * price * (1 - r)           # 退款的不算钱
revenue_old = base_customers * price
delta = revenue_new - revenue_old

print(f"月净收入变化 (来自 {S:,} 个平行宇宙):")
print(f"  期望     = ${delta.mean():>10,.0f}")
print(f"  中位数   = ${np.median(delta):>10,.0f}")
print(f"  95% 区间 = [${np.percentile(delta, 2.5):,.0f}, ${np.percentile(delta, 97.5):,.0f}]")
print(f"  P(亏钱)  = {(delta < 0).mean():.1%}")
print(f"  P(月赚 > $10k) = {(delta > 10_000).mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(delta/1000, bins=120, density=True, alpha=0.75)
ax.axvline(0, color="r", ls="--", label=f"break-even (P(loss)={(delta<0).mean():.1%})")
ax.set_xlabel("Δ revenue ($k / month)"); ax.legend()
ax.set_title("The full decision distribution — not just a point estimate")
plt.tight_layout(); plt.show()


**这就是 MC 的业务终态**：把"拍脑袋点估计"升级成**完整的决策分布**——期望、区间、亏钱概率一次给齐。金融的 VaR、供应链的库存仿真、Uber 的网约车调度模拟全是同款思路。
The business endgame: replacing point guesses with full decision distributions. VaR, inventory sims, dispatch simulations — same move.


<a id="10"></a>
## 10. 小结 + Part 2 总结 🏁

```
MC 恒等式: E_p[f] ≈ (1/N)Σf(xᵢ)   LLN 保证收敛, CLT 给 SE=σ/√N (免费误差条)
  ⭐ SE 与维度无关 → 高维积分唯一可行解

采样工具箱:
  逆变换 X=F⁻¹(U)  ← ppf 的高光时刻
  拒绝采样 (罩子 Mq, 接受率 1/M, 高维死)
  重要性采样 (稀有事件; w=p/q; ESS 诊断; RL off-policy 同款)
  对偶变量 / control variates / 分层 (方差削减全家)
  MCMC: Metropolis 比值只需未归一化密度 → 后验采样 (Part 18 正篇)
```

---

## 🏁 Part 2 全部完成 / Part 2 Complete!

| # | 课 | 核心带走 |
|---|---|---|
| 2.1 | 描述统计 | 稳健性谱系 + n−1 + Anscombe |
| 2.2 | 分布实战 | QQ 图 + floc=0 + AIC 选型 |
| 2.3 | LLN & CLT | SE=σ/√n + 偏态要大 n + Cauchy 惊悚 |
| 2.4 | 抽样 | 偏差加 n 无解 + 分层免费精度 + 水库 |
| 2.5 | 置信区间 | "程序质保"语义 + Wilson + 覆盖率审计 |
| 2.6 | 假设检验 | p~U(0,1) + 永远 Welch + 配对免费午餐 |
| 2.7 | 多重比较 | 1−0.95^m + Bonferroni/Holm vs BH |
| 2.8 | 功效样本量 | n≈16/d² + MDE + 事后 power 谬误 |
| 2.9 | MLE | Hessian→SE + 损失=噪声模型 + censored |
| 2.10 | 贝叶斯 | Beta-Binomial + 正则=先验 + 贝叶斯 A/B |
| 2.11 | Bootstrap | 插件原理 + BCa + 失效清单 |
| 2.12 | 蒙特卡洛 | 期望=平均 + 采样工具箱 + MCMC 预告 |

**三条贯穿线**：
1. **$1/\sqrt{n}$ 无处不在**——SE、MC 误差、样本量公式是同一件事
2. **三种不确定性哲学**：解析（频率公式）/ 先验（贝叶斯）/ 重抽样（bootstrap）——同一问题三把钥匙
3. **模拟是万能审计员**——任何公式存疑就跑覆盖率/功效模拟

### 下一站
**Part 3 · EDA 与数据预处理**——统计武器库就位，开始对真实脏数据动手：缺失值、异常值、特征工程、数据泄漏。
